In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

In [4]:
books = pd.read_parquet("../data/books-final.parquet")
books

,title,subtitle,title_and_subtitle,authors,description,language,isbn13s,tags,year,bx_popularity,gr_popularity,description_source,desc_class
work_id,,,,,,,,,,,,,
OL100003W,Home fires,NaN,Home fires,[Margaret Maron],North Carolina judge Deborah Knott investigate...,en,[9780446608107],"[Arson, Detective And Mystery Stories, Mystery...",2000,8,0,bookcrossing,unique
OL1000053W,No Marriage of Convenience (Avon Romantic Trea...,NaN,No Marriage of Convenience (Avon Romantic Trea...,[Elizabeth Boyle],"Mason St. Clair, the new Earl of Ashlin, has i...",en,[9780380815340],"[Fiction, History, General, Romance]",2000,8,0,bookcrossing,unique
OL1000055W,Once Tempted,NaN,Once Tempted,[Elizabeth Boyle],"Innocence LostA notorious rake, the Marquis of...",en,[9780380815357],"[Spain, Romance, Fiction, General, Man-Woman R...",2001,7,0,bookcrossing,unique
OL1000062W,It takes a hero,NaN,It takes a hero,[Elizabeth Boyle],Rebecca will have to take a page from her own ...,en,[9780060549305],"[Romance, Fiction, General, History, Social Li...",2004,3,0,bookcrossing,unique
OL1000064W,One Night of Passion,NaN,One Night of Passion,[Elizabeth Boyle],They Met At London's Notorious Cyprian's Ball....,en,[9780786249909],"[Romance, Fiction, General, History, Social Li...",2002,16,0,bookcrossing,unique
...,...,...,...,...,...,...,...,...,...,...,...,...,...
unmapped_0820324019,The Unabridged Devil's Dictionary,NaN,The Unabridged Devil's Dictionary,[Ambrose Bierce],If we could only put aside our civil pose and ...,en,[9780820324012],"[Humor, Nonfiction, Classics, Reference, Langu...",1906,1,9087,goodreads,unique
unmapped_155709103X,George Washington's Rules of Civility and Dece...,NaN,George Washington's Rules of Civility and Dece...,[George Washington],Copied out by hand as a young man aspiring to ...,NaN,[9781557091031],[History],1989,1,0,google7k,unique
unmapped_1568821441,The Keeper's Companion,"Blasphemous Knowledge, Forbidden Secrets, and ...","The Keeper's Companion: Blasphemous Knowledge,...",[Keith Herber],"[CALL OF CTHULHU ROLEPLAYING] ""The Keeper's Co...",NaN,[9781568821443],[Games],2000,1,0,google7k,unique


In [6]:
text_to_embed = (
    books["title_and_subtitle"] + ". "
    + books["tags"].map(", ".join) + ". "
    + books["description"]
)
text_to_embed

work_id
OL100003W              Home fires. Arson, Detective And Mystery Stori...
OL1000053W             No Marriage of Convenience (Avon Romantic Trea...
OL1000055W             Once Tempted. Spain, Romance, Fiction, General...
OL1000062W             It takes a hero. Romance, Fiction, General, Hi...
OL1000064W             One Night of Passion. Romance, Fiction, Genera...
                                             ...                        
unmapped_0820324019    The Unabridged Devil's Dictionary. Humor, Nonf...
unmapped_155709103X    George Washington's Rules of Civility and Dece...
unmapped_1568821441    The Keeper's Companion: Blasphemous Knowledge,...
unmapped_1569716110    Domu: A Child's Dream. Manga, Comics, Graphic ...
unmapped_188364254x    Little Novels of Sicily. Short Stories, Classi...
Length: 102117, dtype: str

In [11]:
vectorizer = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5, sublinear_tf=True, dtype=np.float32)
X = vectorizer.fit_transform(text_to_embed)

ids = books.index.to_numpy()
position = {work_id: i for i, work_id in enumerate(ids)}    # work_id -> row number
feature_names = vectorizer.get_feature_names_out()          # column number -> word

assert X.shape[0] == len(ids) == len(books), "vectors and books are out of sync"
print(X.shape)


(102117, 57850)


In [14]:
def find_book(query: str, books: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """Books whose title contains `query` (case-insensitive), most popular first."""
    hits = books[books["title"].str.contains(query, case=False, regex=False, na=False)]
    hits = hits.sort_values(["gr_popularity", "bx_popularity"], ascending=False)
    return hits[["title", "authors", "year", "gr_popularity", "bx_popularity"]].head(n)

In [15]:
def recommend(work_id: str, k: int = 10) -> pd.DataFrame:
    """The k books most similar to `work_id` by TF-IDF cosine similarity."""
    if work_id not in position:
        raise KeyError(f"{work_id} is not in the index (no usable description?)")
    pos = position[work_id]
    similarity_scores = (X @ X[pos].T).toarray().ravel()  # dot product similarity
    similarity_scores[pos] = -1.0  # never recommend the book itself

    top = np.argpartition(-similarity_scores, k)[:k]
    top = top[np.argsort(-similarity_scores[top])]

    result = books.iloc[top][["title", "authors", "tags"]].copy()
    result["tags"] = result["tags"].map(lambda t: list(t)[:5])   # keep the display short
    result.insert(0, "score", similarity_scores[top].round(3))
    return result


#### Try out TF-IDF

In [16]:
find_book("fellowship of the ring", books)

,title,authors,year,gr_popularity,bx_popularity
work_id,,,,,
OL27513W,The Fellowship of the Ring,[J.R.R. Tolkien],1954,2355237,480


In [17]:
find_book("harry potter", books)

,title,authors,year,gr_popularity,bx_popularity
work_id,,,,,
gr:3.Harry_Potter_and_the_Sorcerer_s_Stone,Harry Potter and the Sorcerer's Stone,[J.K. Rowling],1997,7048471,0
gr:136251.Harry_Potter_and_the_Deathly_Hallows,Harry Potter and the Deathly Hallows,[J.K. Rowling],<NA>,2811637,0
OL82536W,Harry Potter and the Prisoner of Azkaban,[J.K. Rowling],1999,2806471,578
OL82537W,Harry Potter and the Chamber of Secrets,[J.K. Rowling],1998,2725847,693
OL82560W,Harry Potter and the Goblet of Fire,[J.K. Rowling],1900,2594622,491
OL82548W,Harry Potter and the Order of the Phoenix,[J.K. Rowling],1903,2507623,447
gr:1.Harry_Potter_and_the_Half_Blood_Prince,Harry Potter and the Half-Blood Prince,[J.K. Rowling],1905,2437658,0
gr:29056083-harry-potter-and-the-cursed-child,Harry Potter and the Cursed Child: Parts One a...,"[Jack Thorne, J.K. Rowling]",2016,701541,0
gr:862041.Harry_Potter_Series_Box_Set,Harry Potter Series Box Set,[J.K. Rowling],<NA>,251756,0


In [21]:
recommend("OL27513W", k=10)

,score,title,authors,tags
work_id,,,,
OL27448W,0.688,The Lord of the Rings,[J.R.R. Tolkien],"[Fantasy, Classics, Fiction, Adventure, Scienc..."
OL27455W,0.531,The Return of the King,[J.R.R. Tolkien],"[Fantasy, Fiction, Classics, Adventure, Scienc..."
g7k:9780739409558,0.429,The Lord of the Rings,[],"[Baggins, Frodo (Fictitious character)]"
gr:30.J_R_R_Tolkien_4_Book_Boxed_Set,0.387,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,[J.R.R. Tolkien],"[Fantasy, Fiction, Classics, Adventure, Scienc..."
OL27479W,0.367,The Two Towers,[J.R.R. Tolkien],"[Fantasy, Fiction, Classics, Adventure, Scienc..."
OL27482W,0.362,The Hobbit,[J.R.R. Tolkien],"[Specimens, Translations Into Russian, Adventu..."
OL27508W,0.289,The Return of the Shadow,"[John Ronald Reuel Tolkien, Christopher Tolkien]","[Fiction, English, 20Th Century, 2001, Middle ..."
OL27506W,0.278,Bilbo's Last Song (At the Grey Havens),[J.R.R. Tolkien],"[English, Posie, Middle Earth Imaginary Place,..."
gr:5907.The_Hobbit_or_There_and_Back_Again,0.276,"The Hobbit, or There and Back Again",[J.R.R. Tolkien],"[Fantasy, Classics, Fiction, Adventure, Young ..."


In [22]:
recommend("OL82536W", k=10)

,score,title,authors,tags
work_id,,,,
OL82548W,0.494,Harry Potter and the Order of the Phoenix,[J.K. Rowling],"[Fantasy, Young Adult, Fiction, Magic, Childrens]"
OL82563W,0.471,Harry Potter and the Philosopher's Stone,[J. K. Rowling],"[Action Adventure, Schools, Adventure Stories,..."
OL82560W,0.433,Harry Potter and the Goblet of Fire,[J.K. Rowling],"[Fantasy, Young Adult, Fiction, Magic, Childrens]"
OL82537W,0.431,Harry Potter and the Chamber of Secrets,[J.K. Rowling],"[Fantasy, Young Adult, Fiction, Magic, Childrens]"
gr:8.Harry_Potter_Boxed_Set_Books_1_5,0.380,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",[J.K. Rowling],"[Fantasy, Young Adult, Fiction, Magic, Adventure]"
OL8460139W,0.376,Harry Potter (series) 1-4,[J. K. Rowling],"[Schools, Childrens Booksages 4-8 Fiction, Pot..."
gr:3.Harry_Potter_and_the_Sorcerer_s_Stone,0.346,Harry Potter and the Sorcerer's Stone,[J.K. Rowling],"[Fantasy, Fiction, Young Adult, Magic, Childrens]"
OL24208534W,0.341,Harry Potter Hogwarts journal,[J. K. Rowling],"[Potter, Fiction, Harry Fictitious Character, ..."
gr:2337379.The_Harry_Potter_trilogy,0.336,The Harry Potter trilogy,[J.K. Rowling],"[Fantasy, Fiction, Young Adult, Childrens, Magic]"


In [23]:
find_book("algorithms", books)

,title,authors,year,gr_popularity,bx_popularity
work_id,,,,,
gr:13542772-automate-this,Automate This: How Algorithms Came to Rule Our...,[Christopher Steiner],2012,3445,0
OL257263W,"The Art of Computer Programming, Volume 1: Fun...",[Donald Ervin Knuth],1973,1722,11
gr:33858244-data-structures-and-algorithms-made-easy,Data Structures and Algorithms Made Easy: Data...,[Narasimha Karumanchi],2011,1252,0
gr:17809497-algorithms-unlocked,Algorithms Unlocked,[Thomas H. Cormen],2013,366,0
gr:33956602-problem-solving-in-data-structures-algorithms-using-python,Problem Solving in Data Structures & Algorithm...,[Hemant Jain],2017,10,0
g7k:9780471738848,Data Structures and Algorithms in Java,"[Michael T. Goodrich, Roberto Tamassia]",2005,0,0
g7k:9780534491826,Data Structures and Algorithms in C++,[Adam Drozdek],2004,0,0


In [24]:
recommend("OL257263W", k=10)

,score,title,authors,tags
work_id,,,,
gr:17809497-algorithms-unlocked,0.381,Algorithms Unlocked,[Thomas H. Cormen],"[Computer Science, Programming, Algorithms, No..."
gr:105099.Effective_Java_,0.262,Effective Java : Programming Language Guide,[Joshua Bloch],"[Programming, Technology, Computer Science, Te..."
g7k:9780534491826,0.262,Data Structures and Algorithms in C++,[Adam Drozdek],[Computers]
OL1970690W,0.247,Structured Computer Organization,[Andrew S. Tanenbaum],"[Computer Science, Programming, Nonfiction, Co..."
gr:33858244-data-structures-and-algorithms-made-easy,0.240,Data Structures and Algorithms Made Easy: Data...,[Narasimha Karumanchi],"[Programming, Algorithms, Computer Science, Te..."
gr:33956602-problem-solving-in-data-structures-algorithms-using-python,0.237,Problem Solving in Data Structures & Algorithm...,[Hemant Jain],[]
OL2966531W,0.234,Patterns of software,[Richard P. Gabriel],"[Computer Programs, Development]"
OL5617116W,0.231,BASIC beginnings,[Susan Drake Lipscomb],"[Computers, Computer Programs, C Computer Prog..."
OL13590735W,0.224,"Introduction to Data Compression, Second Editi...",[Khalid Sayood],[Textbooks]


In [27]:
def explain_match(work_a: str, work_b: str, n: int = 8) -> list[tuple[str, float]]:
    """The words that contribute most to the similarity between two books."""
    a = X[position[work_a]]
    b = X[position[work_b]]
    contribution = a.multiply(b).tocsr()            # element-wise product: non-zero only for shared words
    order = np.argsort(-contribution.data)[:n]      # biggest contributions first
    return [(feature_names[contribution.indices[i]], round(float(contribution.data[i]), 3))
            for i in order]

In [35]:
algorithms = find_book("algorithms", books).index[0]
recs = recommend(algorithms, k=15)
explain_match(algorithms, recs.index[10])  # recs.index[0] returns work id of the first recommended book

[('flash', 0.018),
 ('market', 0.014),
 ('wall', 0.013),
 ('trading', 0.012),
 ('street', 0.01),
 ('journalism', 0.006),
 ('economics', 0.005),
 ('technology', 0.004)]

Now the recommender returns many books by the same author in case of books like LOTR or HP. Return at most 2 books from the same author:

In [36]:
from catalog.normalize import author_key

# main author key per book, aligned with the rows of X (same order as `books`)
main_author = books["authors"].map(lambda a: author_key(a[0]) if len(a) else None).to_numpy()


def recommend(work_id: str, k: int = 10, exclude_same_author: bool = True) -> pd.DataFrame:
    """The k books most similar to `work_id` by TF-IDF cosine similarity."""
    if work_id not in position:
        raise KeyError(f"{work_id} is not in the index (no usable description?)")
    pos = position[work_id]
    similarity_scores = (X @ X[pos].T).toarray().ravel()
    similarity_scores[pos] = -1.0

    if exclude_same_author and main_author[pos] is not None:
        similarity_scores[main_author == main_author[pos]] = -1.0   # also hides other Tolkien books

    top = np.argpartition(-similarity_scores, k)[:k]
    top = top[np.argsort(-similarity_scores[top])]

    result = books.iloc[top][["title", "authors", "tags"]].copy()
    result["tags"] = result["tags"].map(lambda t: list(t)[:5])
    result.insert(0, "score", similarity_scores[top].round(3))
    return result

In [37]:
find_book("lord of the rings", books)

,title,authors,year,gr_popularity,bx_popularity
work_id,,,,,
OL27448W,The Lord of the Rings,[J.R.R. Tolkien],1955,564734,196
gr:30.J_R_R_Tolkien_4_Book_Boxed_Set,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,[J.R.R. Tolkien],1955,110146,0
OL3062780W,The Lord of the Rings: The Making of the Movie...,[Brian Sibley],2001,11051,5
gr:15242.The_Lord_of_the_Rings,The Lord of the Rings: The Art of The Two Towers,[Gary Russell],2003,7661,0
OL16060879W,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...",1969,5875,47
gr:3141295-the-children-of-hurin-the-silmarillion-the-hobbit-the-lord-of-the-rings,The Children of Hurin/The Silmarillion/The Hob...,[J.R.R. Tolkien],2007,502,0
OL14926019W,Novels (Hobbit / Lord of the Rings),[J.R.R. Tolkien],1986,0,11
OL5751219W,The Magical Worlds of the Lord of the Rings,[David Colbert],2002,0,4
OL5753836W,The Lord of the Rings,[Jude Fisher],2001,0,4


In [38]:
recommend("OL27448W", k=10)

,score,title,authors,tags
work_id,,,,
g7k:9780739409558,0.424,The Lord of the Rings,[],"[Baggins, Frodo (Fictitious character)]"
OL4488062W,0.247,The Atlas of Middle-earth,[Karen Wynn Fonstad],"[20Th Century, 2001, Fiction, Middle Earth Ima..."
OL219602W,0.238,The Hobbit: Graphic Novel,[Chuck Dixon],"[Fantasy, Graphic Novels, Fiction, Classics, C..."
OL5099598W,0.222,J.R.R. Tolkien's Hobbit and Lord of the rings,[Anne Pienciak],"[History And Criticism, Middle Earth Imaginary..."
OL6214834W,0.213,Tolkien's Ordinary Virtues,[Mark Eddy Smith],"[2001, Middle Earth Imaginary Place, Fantasy F..."
OL5747799W,0.187,The Lord of the rings,[David Brawn],"[Juvenile Fiction, History And Criticism, Midd..."
g7k:9780618391004,0.185,The Lord of the Rings,[Chris Smith],[Literary Criticism]
OL16456308W,0.184,The Lord of the Rings and Philosophy,"[Gregory Bassham, Eric Bronson]","[Literary Criticism, History And Criticism, Mi..."
OL16060879W,0.183,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...","[Fantasy, Humor, Fiction, Comedy, Science Fict..."
